In [ ]:
# Cell 1: Install and import libraries
!pip install earthengine-api geemap
import ee
import geemap

# Cell 2: Authenticate (one-time setup)
ee.Authenticate()

# Cell 3: Initialize with your project ID
ee.Initialize(project='western-avatar-491910-s2')

# Cell 4: Load Landsat 8 dataset
landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
  .filterDate('2020-01-01', '2024-12-31') \
  .select('ST_B10') \
  .map(lambda image: image.multiply(0.00341802).add(149.0)) # Apply scaling factor and offset to convert to Kelvin

### Visualize the Landsat 8 Image Collection

Now, let's visualize the `landsat8` image collection. We will create a `geemap` interactive map, define visualization parameters for the 'ST_B10' (Surface Temperature) band, and add it to the map.

In [ ]:
# Cell 5: Create a map and add a Landsat 8 image

# Create a geemap interactive map
Map = geemap.Map()

# Define visualization parameters for the 'ST_B10' (Surface Temperature) band
# These values are typical for Landsat 8 surface temperature, adjusted for Kelvin to Celsius (approx)
vis_params = {
    'min': 293.0,  # Adjusted minimum temperature in Kelvin (approx 20°C) for a typical range
    'max': 310.0,  # Adjusted maximum temperature in Kelvin (approx 37°C) for a typical range
    'palette': 'RdYlGn' # Changed to 'RdYlGn' (Red-Yellow-Green) for a more intuitive temperature scale
}

# Get the median image from the collection for visualization
# This will provide a composite image, often reducing cloud cover effects.
first_landsat_image = landsat8.median()

# Check if an image was successfully retrieved
if first_landsat_image:
    # Add the Landsat 8 surface temperature layer to the map
    Map.addLayer(first_landsat_image, vis_params, 'Landsat 8 Surface Temperature (Mumbai)')

    # Center the map to Mumbai
    Map.setCenter(72.8777, 19.0760, 10) # Longitude, Latitude, Zoom Level (centered on Mumbai)

    # Display the map
    display(Map)
else:
    print("No Landsat 8 images found in the specified date range. Please adjust the date filter in Cell 4.")


### Calculate Mean Surface Temperature for a Region of Interest (ROI)

To calculate the mean surface temperature, we first need to define a region of interest. We'll use a rectangular polygon to represent a general area over India, then apply a reducer to get the mean 'ST_B10' (Surface Temperature) value within that region.

In [ ]:
# Define a Region of Interest (ROI) - Example: A small square
# Coordinates are [west, south, east, north]
# This is a much smaller rectangle for demonstration to avoid memory limits.
roi = ee.Geometry.Rectangle([72.7, 18.8, 73.0, 19.2]) # A small area covering Mumbai

# Ensure first_landsat_image is available (from previous cell)
if first_landsat_image:
    # Clip the image to the ROI
    image_clipped = first_landsat_image.clip(roi)

    # Calculate the mean surface temperature within the ROI
    # The 'reducer' specifies the aggregation method (e.g., ee.Reducer.mean() for average)
    # 'scale' is the spatial resolution in meters (e.g., 30m for Landsat).
    # 'crs' specifies the coordinate reference system. 'EPSG:4326' is WGS84.
    mean_temp_kelvin = image_clipped.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=1000, # Increased scale to 1000 meters (1km) to reduce computation time
        crs='EPSG:4326',
        maxPixels=1e9, # Allow a large number of pixels for the calculation
        bestEffort=True # Automatically adjust scale if maxPixels is exceeded
    ).get('ST_B10')

    # Convert the mean temperature from Earth Engine object to a client-side number
    # and then from Kelvin to Celsius
    mean_temp_kelvin_value = mean_temp_kelvin.getInfo()
    if mean_temp_kelvin_value is not None:
        mean_temp_celsius = mean_temp_kelvin_value - 273.15
        print(f"Mean Surface Temperature in ROI (Kelvin): {mean_temp_kelvin_value:.2f} K")
        print(f"Mean Surface Temperature in ROI (Celsius): {mean_temp_celsius:.2f} °C")
    else:
        print("Could not calculate mean temperature. Check ROI or image availability.")
else:
    print("No Landsat 8 image available to calculate mean temperature.")


### Determine Min and Max Temperatures in Mumbai ROI

To create a more effective visualization, let's determine the actual minimum and maximum surface temperatures within our defined Mumbai Region of Interest (ROI). This will help us set more appropriate visualization parameters.

In [ ]:
# Calculate the minimum surface temperature within the ROI
min_temp_kelvin_ee = image_clipped.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=roi,
    scale=1000,
    crs='EPSG:4326',
    maxPixels=1e9,
    bestEffort=True
).get('ST_B10')

# Calculate the maximum surface temperature within the ROI
max_temp_kelvin_ee = image_clipped.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=roi,
    scale=1000,
    crs='EPSG:4326',
    maxPixels=1e9,
    bestEffort=True
).get('ST_B10')

# Get client-side values
min_temp_kelvin_value = min_temp_kelvin_ee.getInfo()
max_temp_kelvin_value = max_temp_kelvin_ee.getInfo()

if min_temp_kelvin_value is not None and max_temp_kelvin_value is not None:
    print(f"Minimum Surface Temperature in Mumbai ROI: {min_temp_kelvin_value:.2f} K ({min_temp_kelvin_value - 273.15:.2f} °C)")
    print(f"Maximum Surface Temperature in Mumbai ROI: {max_temp_kelvin_value:.2f} K ({max_temp_kelvin_value - 273.15:.2f} °C)")
else:
    print("Could not retrieve min/max temperatures. Check image and ROI availability.")


### Visualize Data for Mumbai with Adjusted Range and Legend

Now that we have the actual min and max temperatures for Mumbai, we can refine our visualization parameters (`min` and `max`) to better represent the data. We'll also add a color legend to the map so you can easily understand what each color signifies.

In [ ]:
# Cell 5: Create a map and add a Landsat 8 image (updated with dynamic vis_params and legend)

# Create a geemap interactive map
Map = geemap.Map()

# Define visualization parameters for the 'ST_B10' (Surface Temperature) band
# Use the empirically determined min/max values for a more accurate representation
# Add a small buffer to min/max for better visual spread
if min_temp_kelvin_value is not None and max_temp_kelvin_value is not None:
    dynamic_min = min_temp_kelvin_value - 2 # Subtract a small value
    dynamic_max = max_temp_kelvin_value + 2 # Add a small value
else:
    # Fallback to a sensible default range if min/max couldn't be determined
    dynamic_min = 293.0
    dynamic_max = 310.0

vis_params = {
    'min': dynamic_min,
    'max': dynamic_max,
    'palette': 'RdYlGn' # Red-Yellow-Green for a more intuitive temperature scale
}

# Get the median image from the collection for visualization
first_landsat_image = landsat8.median()

# Check if an image was successfully retrieved
if first_landsat_image:
    # Add the Landsat 8 surface temperature layer to the map
    Map.addLayer(first_landsat_image, vis_params, 'Landsat 8 Surface Temperature (Mumbai)')

    # Center the map to Mumbai
    Map.setCenter(72.8777, 19.0760, 10) # Longitude, Latitude, Zoom Level (centered on Mumbai)

    # Add a legend to the map
    # Correct way to add a colorbar with geemap.Map()
    Map.add_colorbar(
        vis_params,
        label="Surface Temperature (°C)",
        orientation="horizontal",
        position="bottomleft"
    )

    # Display the map
    display(Map)
else:
    print("No Landsat 8 images found in the specified date range. Please adjust the date filter in Cell 4.")